# Review Text — aggregated per short-stay listing

**Sources:**
- `analysis/short_stay_listing_features.csv` — the short-stay listing IDs (1 row/listing, defines scope)
- `raw data/reviews.csv` — 1 row per individual review

Kept as its own notebook/dataset (not merged into `short_stay_listing_features.csv`)
because review text is long and variable-length — joining it into the listing
table would mean rewriting price/host/description data every time review text
changes, and one popular listing's reviews alone can run 50K+ characters.

This notebook collapses many review rows into **one row per listing**
(same grain as the feature table, so it's a clean 1:1 join later on
`listing_id`) and saves the result to `analysis/short_stay_review_text.csv`.

In [1]:
import re
import pandas as pd

pd.set_option("display.max_colwidth", 120)

## 1. Scope: short-stay listing IDs

In [2]:
listing_ids = pd.read_csv("short_stay_listing_features.csv", usecols=["id"])["id"].unique()
print(f"Short-stay listings in scope: {len(listing_ids):,}")

Short-stay listings in scope: 1,483


## 2. Load reviews and clean the text

Spot-checking the raw comments showed HTML artifacts (`<br/>` tags) and
inconsistent whitespace — strip those before aggregating.

In [3]:
reviews = pd.read_csv("../raw data/reviews.csv", usecols=["listing_id", "date", "comments"], parse_dates=["date"])
reviews = reviews[reviews["listing_id"].isin(listing_ids)].dropna(subset=["comments"]).copy()

def clean_comment(text):
    text = re.sub(r"<br\s*/?>", " ", text)      # strip <br/> tags
    text = re.sub(r"<[^>]+>", " ", text)          # strip any other HTML tags
    text = re.sub(r"\s+", " ", text).strip()      # collapse whitespace/newlines
    return text

reviews["comments_clean"] = reviews["comments"].apply(clean_comment)

print(f"Reviews in scope (short-stay listings, non-null comments): {len(reviews):,}")
reviews[["listing_id", "date", "comments_clean"]].head()

Reviews in scope (short-stay listings, non-null comments): 24,476


,listing_id,date,comments_clean
2379,4499317,2014-11-15,Great location and every stuff you need for short business trip is ready and it's very convenient. I would recommend...
2380,4499317,2014-12-05,"Stephanie and team were great following-up via email and with details about the listing (location, directions etc). ..."
2381,4499317,2015-01-08,"I love this apartment very much. It is convenient for you to catch MRT. Near this apartment, you can find Robinson f..."
2382,4499317,2015-02-13,"Thanks to Stephanie and her staff for receiving me and making me comfortable during my stay. The loft is compact, in..."
2383,4499317,2015-03-31,Stephanie was able to accommodate to guest's requirements. We had to check in at 530 am but she found solutions to m...


## 3. Cap and aggregate — most recent 20 reviews per listing

Keeps text length bounded (a listing with hundreds of reviews doesn't dwarf
one with a handful) and favors recent guest experience over old reviews.

In [4]:
REVIEWS_PER_LISTING_CAP = 20

capped = (
    reviews.sort_values(["listing_id", "date"], ascending=[True, False])
           .groupby("listing_id")
           .head(REVIEWS_PER_LISTING_CAP)
)

review_text = (
    capped.groupby("listing_id")
          .agg(review_text=("comments_clean", lambda c: " ".join(c)),
               reviews_used=("comments_clean", "count"),
               latest_review_used=("date", "max"))
          .reset_index()
)
review_text["review_text_length"] = review_text["review_text"].str.len()

print(f"Listings with aggregated review text: {len(review_text):,} / {len(listing_ids):,}")
review_text.describe(include="all")

Listings with aggregated review text: 950 / 1,483


,listing_id,review_text,reviews_used,latest_review_used,review_text_length
count,9.500000e+02,950,950.000000,950,950.000000
unique,NaN,950,NaN,NaN,NaN
top,NaN,"Das Haus war leicht zu finden, das Zimmer selbst war super sauber und alles in Ordnung, daher empfehlenswert. Wie an...",NaN,NaN,NaN
freq,NaN,1,NaN,NaN,NaN
mean,8.771069e+17,NaN,8.574737,2025-08-28 22:29:03.157894,1662.304211
min,4.499317e+06,NaN,1.000000,2017-02-27 00:00:00,2.000000
25%,5.277068e+07,NaN,2.000000,2025-07-07 06:00:00,271.250000
50%,1.131765e+18,NaN,6.000000,2026-02-06 00:00:00,992.000000
75%,1.394699e+18,NaN,17.750000,2026-05-15 18:00:00,2812.500000
max,1.675861e+18,NaN,20.000000,2026-06-28 00:00:00,8307.000000


## 4. Reindex to the full listing scope

Listings with zero reviews get an empty `review_text` (not dropped) — keeps
this file at the exact same grain (1 row per short-stay listing) as
`short_stay_listing_features.csv`, so the later join is a clean 1:1 merge.

In [5]:
review_text_full = (
    pd.DataFrame({"listing_id": listing_ids})
      .merge(review_text, on="listing_id", how="left")
)
review_text_full["reviews_used"] = review_text_full["reviews_used"].fillna(0).astype(int)
review_text_full["review_text"] = review_text_full["review_text"].fillna("")
review_text_full["review_text_length"] = review_text_full["review_text_length"].fillna(0).astype(int)

print(f"Final shape: {review_text_full.shape}  (matches {len(listing_ids):,} short-stay listings)")
print(f"Listings with 0 reviews (empty review_text): {(review_text_full['reviews_used'] == 0).sum():,}")
review_text_full.head()

Final shape: (1483, 5)  (matches 1,483 short-stay listings)
Listings with 0 reviews (empty review_text): 533


,listing_id,review_text,reviews_used,latest_review_used,review_text_length
0,4499317,"Das Haus war leicht zu finden, das Zimmer selbst war super sauber und alles in Ordnung, daher empfehlenswert. Wie an...",20,2026-05-23,5866
1,4541183,Support from Mary and rome was super helpful. From check in to checkout it was a too smooth process. Property is at ...,20,2026-06-26,5419
2,4973227,"Clean, well laid out, not noisy (maybe I got lucky with a room at the back on level 3, not on the river) My only com...",20,2026-02-07,5624
3,5052014,"its a great place in perfect location, less than 10 minutes to chinatown and mrt its was very clean and comfortable ...",16,2026-02-18,2981
4,5233415,"I stayed at Triple 2 Studio for +50 days and really enjoyed my stay. The location was perfect, situated between Outr...",20,2026-06-09,5691


## 5. Save

In [6]:
review_text_full.to_csv("short_stay_review_text.csv", index=False)
print(f"Saved {len(review_text_full):,} rows to analysis/short_stay_review_text.csv")

Saved 1,483 rows to analysis/short_stay_review_text.csv
